In [42]:
from sqlalchemy import create_engine
import io
import os
from pathlib import Path
import yaml
import pandas as pd
import psycopg2

# read credentials stored in yaml file into data dictionary
searching_file = 'credentials.yml'
lookup_folder=r'C:\MySpace\Working\DE01\data_engineering_practices'


def lookup_file_path (searching_file, lookup_folder):
    for root, dirs, files in os.walk(lookup_folder):
        for name in files:
            if name == searching_file:  
                return os.path.abspath(os.path.join(root, name))
            

# read credentials stored in yaml file into data dictionary
config_file_path = lookup_file_path(searching_file,lookup_folder)
# print(config_file_path)
with open(config_file_path, 'r') as f:
    try:
        config_data = yaml.safe_load(f)
        # print(config_data)
    except yaml.YAMLError as exc:
        print(exc)

# get Credentials
# Database credentials
user = config_data['postgresql']['username']
password = config_data['postgresql']['password']
host = 'cm-de-k1-db.c9ms60q6cw37.ap-southeast-2.rds.amazonaws.com'
port = '5432'
database = 'postgres'


# Construct the connection string
# The 'postgresql+psycopg2' part specifies the dialect and driver
connection_string = f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}'

# Create the SQLAlchemy engine
engine = create_engine(connection_string)
connection = engine.connect()


Read data 

In [44]:
# HINTS
# 1. Read data from SQL table coffee.sales
# 2. Read all data and export to 
schema = 'coffee'
table = 'sales'
reading_table='coffee.sales'

"""Fetches data into a pandas sDataFrame."""
try:
    # Use pandas to read the SQL query results into a DataFrame]
    query=f"Select * from {schema}.{table}"
    df = pd.read_sql_query(query, con=engine)
    
    print("Downloaded data into a pandas DataFrame")
except Exception as e:
    print(f"Error fetching data with pandas: {e}")


print(df.head(5))
  


Downloaded data into a pandas DataFrame
         date                datetime cash_type                 card  money  \
0  2024-03-01 2024-03-01 10:15:50.520      card  ANON-0000-0000-0001   38.7   
1  2024-03-01 2024-03-01 12:19:22.539      card  ANON-0000-0000-0002   38.7   
2  2024-03-01 2024-03-01 12:20:18.089      card  ANON-0000-0000-0002   38.7   
3  2024-03-01 2024-03-01 13:46:33.006      card  ANON-0000-0000-0003   28.9   
4  2024-03-01 2024-03-01 13:48:14.626      card  ANON-0000-0000-0004   38.7   

     coffee_name  
0          Latte  
1  Hot Chocolate  
2  Hot Chocolate  
3      Americano  
4          Latte  


Load to storage

In [ ]:
import sqlite3
# Full load

# create destination database
dest_database_file = "destination.db"
dest_conn = sqlite3.connect(dest_database_file)# database file will be created if it does not exist

# 1. Upload data to a table named: coffee_sales
# Write the DataFrame to a new table named 'coffee_sales'
# if_exists='replace': creates a new table, dropping any existing one with the same name
# index=False: prevents the pandas DataFrame index from being written as a column in the SQL table
df.to_sql(name='coffee_sales', con=dest_conn, if_exists='replace', index=False)

# Close the connection
dest_conn.close()

Incremental Loading

In [ ]:
# 1. Read latest data from storage
# 2. Filter incremental records from source
# 3. Load as 'append' to existing table